## Preprocess Data

In [4]:
# %%writefile ../build/songs_preprocess.py

"""
songs_preprocess.py
-------------------
ETL pipeline for FMA-small dataset.

Goal:
- Normalize raw music metadata
- Resolve genres into human-readable format
- Build clean dataset for downstream RAG / search / analytics

Output:
- ../cleaned_data/songs_clean.csv
"""

import pandas as pd
import ast


def build_songs_clean_csv():

    # =========================================================
    # STEP 1: LOAD RAW DATASETS
    # =========================================================
    # Each file contains a different slice of the music graph:
    # - tracks: core metadata (title, artist, album)
    # - features: audio features (MFCC, spectral, etc.)
    # - echonest: additional metadata enrichment
    # - genres: mapping from genre_id → genre_name

    features = pd.read_csv(
        "../data/fma-small/features.csv",
        skiprows=[0, 1, 2],
        low_memory=False
    )

    echonest_raw = pd.read_csv(
        "../data/fma-small/echonest.csv",
        low_memory=False
    )

    tracks = pd.read_csv(
        "../data/fma-small/tracks.csv",
        header=[0, 1],
        low_memory=False
    )

    genres = pd.read_csv(
        "../data/fma-small/genres.csv",
        low_memory=False
    )

    # =========================================================
    # STEP 2: NORMALIZE ECHONEST STRUCTURE
    # =========================================================
    # Fix broken header structure and standardize track_id column

    echonest_raw.columns = echonest_raw.iloc[1]
    echonest = echonest_raw.drop([0, 1])
    echonest.rename(columns={echonest.columns[0]: "track_id"}, inplace=True)

    # =========================================================
    # STEP 3: FLATTEN MULTI-INDEX TRACK COLUMNS
    # =========================================================
    # Convert hierarchical columns into flat names:
    # ('track', 'title') → 'track_title'

    tracks.columns = ['_'.join(col).strip() for col in tracks.columns.values]
    tracks.rename(columns={tracks.columns[0]: "track_id"}, inplace=True)
    tracks = tracks.drop(index=0)

    # =========================================================
    # STEP 4: ENSURE CONSISTENT ID TYPES
    # =========================================================
    # Prevent merge issues caused by string/int mismatches

    for df in [tracks, features, echonest]:
        df["track_id"] = pd.to_numeric(df["track_id"], errors="coerce")

    # =========================================================
    # STEP 5: PARSE GENRE LISTS SAFELY
    # =========================================================
    # Convert stringified lists → Python lists

    tracks["track_genres_all"] = tracks["track_genres_all"].apply(
        lambda x: ast.literal_eval(x) if pd.notnull(x) else []
    )

    # =========================================================
    # STEP 6: EXPLODE GENRES (NORMALIZATION STEP)
    # =========================================================
    # Convert:
    # [genre1, genre2] → row per genre

    tracks_exploded = tracks.explode("track_genres_all")

    # =========================================================
    # STEP 7: MAP GENRE IDs → HUMAN READABLE NAMES
    # =========================================================

    tracks_genres_named = tracks_exploded.merge(
        genres,
        left_on="track_genres_all",
        right_on="genre_id",
        how="left"
    )

    # =========================================================
    # STEP 8: RE-GROUP GENRES PER TRACK
    # =========================================================
    # Reconstruct list of genre names per track_id

    tracks_grouped = (
        tracks_genres_named.groupby("track_id")["title"]
        .apply(list)
        .reset_index()
        .rename(columns={"title": "genre_list"})
    )

    # =========================================================
    # STEP 9: MERGE BACK INTO CORE TRACK DATA
    # =========================================================

    tracks_named = tracks.merge(tracks_grouped, on="track_id", how="left")

    # =========================================================
    # STEP 10: MERGE FEATURE + ENRICHMENT DATA
    # =========================================================

    songs_enriched = tracks_named.merge(features, on="track_id", how="left")
    songs_enriched = songs_enriched.merge(echonest, on="track_id", how="left")

    # =========================================================
    # STEP 11: RESOLVE ARTIST NAME CLEANLY
    # =========================================================
    # Prefer echonest value, fallback to tracks dataset

    songs_enriched["artist_name"] = songs_enriched["artist_name_x"].fillna(
        songs_enriched["artist_name_y"]
    )
    songs_enriched["artist_name"] = songs_enriched["artist_name"].fillna("Unknown Artist")

    songs_enriched.drop(columns=["artist_name_x", "artist_name_y"], inplace=True)

    # =========================================================
    # STEP 12: SELECT FINAL CLEAN SCHEMA
    # =========================================================
    # Keep only fields needed for downstream usage

    songs_final = songs_enriched[
        ["track_id", "track_title", "artist_name", "album_title", "genre_list"]
    ].copy()

    # =========================================================
    # STEP 13: CLEAN + SERIALIZE GENRE LIST
    # =========================================================
    # Convert:
    # list → "genre1, genre2"
    # Ensure no NaN contamination

    def safe_join_genres(x):
        if isinstance(x, list):
            return ", ".join(str(g) for g in x if pd.notna(g))
        return ""

    songs_final["genre_list"] = songs_final["genre_list"].apply(safe_join_genres)

    # =========================================================
    # STEP 14: SAVE CLEAN DATASET
    # =========================================================

    output_path = "../cleaned_data/songs_clean.csv"
    songs_final.to_csv(output_path, index=False)

    print(f"✅ Cleaned songs dataset saved at {output_path}")
    print(f"🎧 Total songs processed: {len(songs_final)}")

    return songs_final


if __name__ == "__main__":
    build_songs_clean_csv()

Writing ../build/songs_preprocess.py


In [1]:
%%writefile ../build/songs_index_build.py

"""
songs_index_build.py
--------------------
Build FAISS vector index from cleaned FMA-small songs dataset.

Purpose:
- Load cleaned song metadata (output of preprocessing module)
- Convert each song into a semantic text document
- Generate embeddings using HuggingFace sentence-transformers
- Store vectors in FAISS for fast similarity search (RAG-ready retrieval layer)

Input:
- ../cleaned_data/songs_clean.csv

Output:
- ../vectorstores/faiss_songs_index (FAISS vector store)
"""

import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os


def build_songs_index():

    # =========================================================
    # STEP 1: LOAD CLEANED DATASET
    # =========================================================
    # We assume preprocessing has already:
    # - resolved genres
    # - normalized missing values
    # - flattened metadata into clean columns

    songs = pd.read_csv("../cleaned_data/songs_clean.csv")

    # =========================================================
    # STEP 2: CONVERT EACH SONG → NATURAL LANGUAGE DOCUMENT
    # =========================================================
    # Why?
    # Embedding models perform best on semantic text, not tabular data.
    # So we transform structured rows into descriptive "mini-documents".

    song_docs = []

    for _, row in songs.iterrows():

        # Create a human-readable semantic representation of a song
        # This helps embeddings capture relationships like:
        # artist ↔ genre ↔ album ↔ track identity

        text = f"""
        Title: {row['track_title']}. 
        Performed by {row['artist_name']}. 
        Released under the album {row['album_title']}. 
        Its genres include {row['genre_list']}.
        """

        # =========================================================
        # STEP 2.1: CLEAN WHITESPACE NOISE
        # =========================================================
        # Removes:
        # - newlines
        # - multiple spaces
        # - indentation artifacts from f-string formatting
        #
        # This ensures consistent tokenization for embeddings.

        text = " ".join(text.split())

        # Wrap into LangChain Document format
        song_docs.append(Document(page_content=text))

    print(f"✅ Prepared {len(song_docs)} song documents.")

    # =========================================================
    # STEP 3: INITIALIZE EMBEDDING MODEL
    # =========================================================
    # Using a lightweight, high-performance transformer model
    # suitable for semantic similarity tasks.

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # =========================================================
    # STEP 4: BUILD FAISS VECTOR INDEX
    # =========================================================
    # FAISS converts embeddings into a fast similarity search index.
    # This enables:
    # - nearest-neighbor search
    # - semantic retrieval
    # - foundation for RAG systems

    vectorstore = FAISS.from_documents(song_docs, embeddings)

    # =========================================================
    # STEP 5: SAVE INDEX TO DISK
    # =========================================================
    # Persist vector database so it can be reused without recomputation

    output_path = "../vectorstores/faiss_songs_index"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    vectorstore.save_local(output_path)

    print(f"✅ Songs FAISS index built and saved at {output_path}")


# =========================================================
# ENTRY POINT
# =========================================================
# Allows script execution directly from terminal or notebook

if __name__ == "__main__":
    build_songs_index()

Overwriting ../build/songs_index_build.py


## Validate Data

In [ ]:
# Total rows
total_rows = len(songs_final)

# Count missing and non-missing per column
missing_count = songs_final.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)


## Load Variables

In [8]:
# load API key from .env 
from dotenv import load_dotenv, find_dotenv 
load_dotenv(find_dotenv())
# llm model
model="gpt-4o-mini"

## Business Logic

In [10]:
# %%writefile ../src/chain_songs.py
"""
chain_songs.py
----------------
Core LangChain module for SongsRAG.

Responsibilities:
- Load the persisted FAISS index (built once in build/songs_build.py)
- Define a retrieval-augmented generation (RAG) chain for song queries
- Designed for integration into the CultRAG pipeline
"""

# --- BUSINESS LOGIC ---

# Core LangChain modules
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Embeddings + Vectorstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from utils.paths import VECTORSTORES_DIR

# Step 1: Load persisted FAISS index (built once in build/songs_build.py)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    str(VECTORSTORES_DIR / "faiss_songs_index"),
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever()

# Step 2: LLM → define the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Step 3: Prompt Template → instructions for how the assistant should answer
prompt_songs = ChatPromptTemplate.from_template("""
You are a song recommendation assistant using retrieval-augmented generation (RAG).

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "Not found in the catalog."

Conversation so far:
{history}

Context:
{context}

Question:
{question}

Rules:
- ONLY include SONGS in your answer.
- DO NOT include books, movies, or any other media, even if present in the context.
- Ignore any non-song entries in the context completely.
- Do NOT add items that are not in the context.
- Do NOT guess or hallucinate.
- Do NOT mention other domains (books, movies) or their absence.
- If context contains mixed types, extract and return ONLY song-related entries.
- Output a valid markdown table with headers.
- Include a short summary after the table.

Answer:
""")

# Step 4: Helper to format retrieved docs into a single string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single string.
    Each document’s page_content is joined with double newlines.
    Used to feed retrieved context into the prompt.
    """
    return "\n\n".join([d.page_content for d in docs])

# Step 5: Build the Core LCEL Retrieval Chain
chain_songs = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "history": lambda x: x.get("history", "")
    }
    | prompt_songs
    | llm
)


Overwriting ../src/chain_songs.py


## Query & Display

In [ ]:
# Display helpers for Jupyter
from IPython.display import display, Markdown

# --- 8. Query ---
query = "Please list 3 best party songs"

response = chain_songs.invoke({"question": query})

# --- 9. Display ---
display(Markdown(response.content))